# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR^2](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant


## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset Title: {dataset.metadata.name}\n\nDescription: {dataset.metadata.description}")


## 2. Data Overview
Let's review the available record sets, fields, and their `@id`s. This helps to identify the structure and contents of the data.


In [ ]:
# List record sets and their fields by @id
print("Available record sets in the dataset:")
for rs in dataset.metadata.record_sets:
    print(f"- Record set name: {rs.name} | Record set @id: {rs.id}")
    print("  Fields:")
    for field in rs.fields:
        col_names = [col.name for col in (field.columns or [])]
        print(f"    - Field name: {field.name} | Field @id: {field.id} | Data type: {field.data_type} | Columns: {col_names}")
    print()


## 3. Data Extraction
Below, we extract all records from each record set, using the `@id` fields as identifiers. Each extracted table is loaded into a separate pandas DataFrame, for easy analysis.

For demonstration, the record set with `@id` of `'https://api.app.sen.science/frontiers/7154140/ffef04c5-bcc4-4c6b-bce2-345801507376'` (the primary protein record set) will be used for further analysis.


In [ ]:
# Gather all record sets' @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
# Dict to hold DataFrames for each record set
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)
    print(f"Loaded record set {record_set_id} with shape: {dataframes[record_set_id].shape}")

# Inspect columns of a primary record set for demonstration:
main_rs_id = record_set_ids[0]  # Replace index if necessary after inspecting the above overview
print(f"\nColumns in primary record set (@id={main_rs_id}):\n{dataframes[main_rs_id].columns.tolist()}")
dataframes[main_rs_id].head()


## 4. Exploratory Data Analysis (EDA)
We'll perform data filtering, normalization, and simple grouping below. Please refer to the earlier overview for valid field (column) `@id` references.

For this demonstration, we select the `cr:coveragePercentage` field, which likely captures protein coverage percentage (make sure to confirm this with the field list from the previous overview), and group by the `cr:description` field.

In [ ]:
# Change these field IDs as appropriate based on your dataset's field listing
numeric_field = 'cr:coveragePercentage'  # Use the full '@id' as printed above
group_field = 'cr:description'

# Check if the numeric field exists
if numeric_field in dataframes[main_rs_id].columns:
    df = dataframes[main_rs_id].copy()
    # Ensure numeric conversion for EDA
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = 10
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered records with {numeric_field} > {threshold}:")
    print(filtered_df[[numeric_field, group_field]].head())

    # Normalize the numeric field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records:")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Group by a category field
    if group_field in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
        print(f"\nMean {numeric_field} by group ({group_field}):")
        print(grouped_df.head())
else:
    print(f"Field '{numeric_field}' not found in DataFrame columns: {dataframes[main_rs_id].columns.tolist()}")


## 5. Visualization
Let's visualize the distribution of protein coverage percentage for the top categories.


In [ ]:
import matplotlib.pyplot as plt

if numeric_field in dataframes[main_rs_id].columns:
    plt.figure(figsize=(8,5))
    # Histogram of the numeric field
    data = dataframes[main_rs_id][numeric_field].dropna().astype(float)
    plt.hist(data, bins=30, edgecolor='k', alpha=0.7)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel("Coverage Percentage")
    plt.ylabel("Frequency")
    plt.show()

    # Boxplot by group (top 5 groups only)
    if group_field in dataframes[main_rs_id].columns:
        top_groups = dataframes[main_rs_id][group_field].value_counts().index[:5]
        plt.figure(figsize=(10,6))
        df_box = dataframes[main_rs_id][dataframes[main_rs_id][group_field].isin(top_groups)]
        df_box.boxplot(column=numeric_field, by=group_field, rot=60)
        plt.title(f"{numeric_field} by {group_field} (top 5)")
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel("Coverage Percentage")
        plt.show()


## 6. Conclusion
In this notebook, we:
- Loaded and explored the metadata and records of the FAIR^2 dataset using `mlcroissant`
- Identified key record sets and fields by their `@id`
- Performed basic exploratory analysis on the protein coverage data
- Visualized distributions and groupings within the main data table

This flexible pipeline can be adapted to other Croissant datasets—simply update the schema URL and adjust field names/IDs as appropriate.
